# Mapa de Geoanalitica Territorial - ITT (Ciudad Paraiso / Pulmon de Oriente)

Notebook de geoanalitica territorial. La unidad espacial de analisis es la **manzana catastral** (`data/catastral/MANZANAS_U_2024.shp`). El pipeline:

1. Carga capas catastrales reales (manzanas, terrenos, construcciones) desde `data/catastral/`.
2. Calcula indicadores por manzana e integra los eventos ITT (seguridad / ambiente).
3. Clasifica cada manzana en un nivel ITT (Activacion / Consolidacion / Transformacion / Escala).
4. Exporta artefactos intermedios a `data_processed_geo/` (`.geojson`, `.parquet`).
5. Genera un mapa `.html` interactivo y autonomo (Folium) con popups enriquecidos.

**Convenciones del proyecto:** entorno unico `uv`, codigo sin emojis, paleta Okabe-Ito accesible para daltonismo, CRS metrico EPSG:9377 y capa web EPSG:4326.

> IMPORTANTE - datos de eventos: hurtos, homicidios, rinas, lesionados, siniestralidad, SPA, VIF y censo arboreo NO estan aun en el proyecto. La celda 4 genera datos **sinteticos marcados como placeholder** para que el pipeline sea ejecutable de punta a punta. Al llegar los datos reales, se reemplaza unicamente esa celda.

## 1. Entorno y paleta Okabe-Ito

In [ ]:
# Dependencias: ejecutar en el entorno uv del proyecto.
#   uv run --with geopandas --with folium --with pyarrow jupyter lab
# o, para una celda suelta: uv run --with geopandas --with folium --with pyarrow python ...
import os
import math
import json
import numpy as np
import pandas as pd
import geopandas as gpd

# Paleta Okabe-Ito (accesible para daltonismo) - misma paleta que el resto del proyecto
OKI_AZUL     = "#0072B2"
OKI_NARANJA  = "#E69F00"
OKI_VERDE    = "#009E73"
OKI_AMARILLO = "#F0E442"
OKI_AZUL_C   = "#56B4E9"
OKI_BERMELL  = "#D55E00"
OKI_MORADO   = "#CC79A7"
OKI_GRIS     = "#333333"
BG = "#F4F6F9"

# Niveles ITT (color saturado y su version pastel para bandas / coropleta suave)
NIVEL_COLORS = {
    "Activacion":     OKI_BERMELL,
    "Consolidacion":  OKI_NARANJA,
    "Transformacion": OKI_VERDE,
    "Escala":         OKI_AZUL,
}
NIVEL_PASTEL = {
    "Activacion":     "#FFCDD2",
    "Consolidacion":  "#FFE0B2",
    "Transformacion": "#C8E6C9",
    "Escala":         "#BBDEFB",
}
# Color tematico por indicador de evento
EVENTO_COLORS = {
    "hurtos":        OKI_AZUL,
    "siniestros":    OKI_NARANJA,
    "rinas":         OKI_VERDE,
    "spa":           OKI_AMARILLO,
    "lesionados":    OKI_AZUL_C,
    "homicidios":    OKI_BERMELL,
    "vif":           OKI_MORADO,
}

# CRS del proyecto
CRS_WEB    = 4326   # WGS84, capa web
CRS_METRIC = 9377   # MAGNA-SIRGAS Origen Nacional, calculo metrico

print("Entorno listo. Paleta Okabe-Ito cargada.")

## 2. Rutas y carga de capas catastrales (datos reales)

In [ ]:
# Resolucion robusta de rutas. Funciona en local (uv) y en Google Colab.
# Se puede forzar con la variable de entorno CATASTRAL_DIR.
REQUERIDOS = ["MANZANAS_U_2024.shp", "TERRENOS_U_2024.shp", "CONSTRUCCION_U_2024.shp"]

def tiene_shapefiles(p):
    return bool(p) and all(os.path.isfile(os.path.join(p, f)) for f in REQUERIDOS)

def resolver_catastral():
    # 1) Variable de entorno explicita
    env = os.environ.get("CATASTRAL_DIR")
    candidatos = [env] if env else []
    # 2) Rutas locales (notebook en notebooks/, data un nivel arriba)
    candidatos += [
        os.path.join("..", "data", "catastral"),
        os.path.join("data", "catastral"),
        "catastral",
    ]
    # 3) Google Colab con Drive montado (ajusta la ruta a tu carpeta en Drive)
    candidatos += [
        "/content/data/catastral",
        "/content/drive/MyDrive/mapa_Analitico_ciudad_Paraiso/data/catastral",
    ]
    for p in candidatos:
        if tiene_shapefiles(p):
            return os.path.abspath(p)
    return None

CATASTRAL_DIR = resolver_catastral()

if CATASTRAL_DIR is None:
    # En Colab: ofrecer subir los shapefiles (los 6+ archivos por capa: shp, dbf, shx, prj, cpg...).
    try:
        from google.colab import files
        destino = "/content/data/catastral"
        os.makedirs(destino, exist_ok=True)
        print("No se encontro data/catastral. Sube los archivos de las 3 capas")
        print("(cada capa: .shp .dbf .shx .prj .cpg ...). Selecciona todos a la vez:")
        subidos = files.upload()
        for nombre, contenido in subidos.items():
            with open(os.path.join(destino, nombre), "wb") as fh:
                fh.write(contenido)
        CATASTRAL_DIR = destino if tiene_shapefiles(destino) else None
    except ImportError:
        pass

if CATASTRAL_DIR is None:
    raise FileNotFoundError(
        "No se encontraron los shapefiles catastrales (" + ", ".join(REQUERIDOS) + ").\n"
        "Local: ejecuta desde la carpeta notebooks/ del proyecto (data/catastral/ debe existir un nivel arriba).\n"
        "Colab: monta Google Drive y define CATASTRAL_DIR, o sube los archivos cuando se solicite.\n"
        "Tambien puedes fijar la ruta: os.environ['CATASTRAL_DIR'] = '/ruta/a/data/catastral'"
    )

# Carpeta de salida junto a la raiz del proyecto (o local al notebook en Colab)
OUT_DIR = os.path.abspath(os.path.join(os.path.dirname(CATASTRAL_DIR.rstrip(os.sep)), "..", "data_processed_geo"))
os.makedirs(OUT_DIR, exist_ok=True)

mz = gpd.read_file(os.path.join(CATASTRAL_DIR, "MANZANAS_U_2024.shp"))
terrenos = gpd.read_file(os.path.join(CATASTRAL_DIR, "TERRENOS_U_2024.shp"))
construcciones = gpd.read_file(os.path.join(CATASTRAL_DIR, "CONSTRUCCION_U_2024.shp"))

# Normalizar clave de union. CONSTRUCCION no trae IDMANZANA: se compone
# como COMUNA(2) + BARRIO(2) + MANZANA(4), igual que en MANZANAS y TERRENOS.
def asegurar_idmanzana(g):
    if "IDMANZANA" not in g.columns:
        g["IDMANZANA"] = (g["COMUNA"].astype(str) + g["BARRIO"].astype(str) + g["MANZANA"].astype(str))
    g["IDMANZANA"] = g["IDMANZANA"].astype(str).str.strip()
    return g

mz = asegurar_idmanzana(mz)
terrenos = asegurar_idmanzana(terrenos)
construcciones = asegurar_idmanzana(construcciones)

print("Catastral:", CATASTRAL_DIR)
print("Salida procesada:", OUT_DIR)
print(f"Manzanas: {len(mz)} | CRS: {mz.crs}")
print(f"Terrenos (predios): {len(terrenos)}")
print(f"Construcciones: {len(construcciones)}")

# Poligono de la zona de intervencion (Ciudad Paraiso). Se resuelve igual que la
# data catastral (local + Colab) y se reproyecta a WGS84 para los mapas.
def resolver_poligono():
    env = os.environ.get("POLIGONO_ZONA")
    cand = [env] if env else []
    cand += [
        os.path.join(os.path.dirname(CATASTRAL_DIR.rstrip(os.sep)), "geojson", "poligono_ciudad_Paraiso.geojson"),
        os.path.join("..", "data", "geojson", "poligono_ciudad_Paraiso.geojson"),
        os.path.join("data", "geojson", "poligono_ciudad_Paraiso.geojson"),
        "/content/data/geojson/poligono_ciudad_Paraiso.geojson",
        "/content/drive/MyDrive/mapa_Analitico_ciudad_Paraiso/data/geojson/poligono_ciudad_Paraiso.geojson",
    ]
    for p in cand:
        if p and os.path.isfile(p):
            return os.path.abspath(p)
    return None

_poly_path = resolver_poligono()
POLIGONO_ZONA = None
if _poly_path:
    POLIGONO_ZONA = gpd.read_file(_poly_path).to_crs(CRS_WEB)
    print("Poligono de zona:", _poly_path, "| features:", len(POLIGONO_ZONA))
else:
    print("Aviso: no se encontro poligono_ciudad_Paraiso.geojson; los mapas se dibujan sin el contorno de zona.")

## 3. Indicadores reales por manzana

In [ ]:
# Area de manzana en metros (reproyectada a CRS metrico)
mz_m = mz.to_crs(CRS_METRIC)
mz = mz.copy()
mz["area_m2"] = mz_m.geometry.area.values

# Predios por manzana
pred_por_mz = terrenos.groupby("IDMANZANA").size().rename("num_predios")

# Construcciones: numero y pisos promedio por manzana
construcciones["NPISOS"] = pd.to_numeric(construcciones.get("NPISOS"), errors="coerce")
const_agg = construcciones.groupby("IDMANZANA").agg(
    num_construcciones=("NPISOS", "size"),
    pisos_prom=("NPISOS", "mean"),
    pisos_max=("NPISOS", "max"),
)

mz = mz.merge(pred_por_mz, on="IDMANZANA", how="left")
mz = mz.merge(const_agg, on="IDMANZANA", how="left")
mz[["num_predios", "num_construcciones"]] = mz[["num_predios", "num_construcciones"]].fillna(0).astype(int)
mz["pisos_prom"] = mz["pisos_prom"].fillna(0).round(2)
mz["pisos_max"] = mz["pisos_max"].fillna(0).astype(int)

# Densidad predial (predios por hectarea)
mz["densidad_predial"] = (mz["num_predios"] / (mz["area_m2"] / 10000.0)).round(1)

mz[["IDMANZANA", "COMUNA", "BARRIO", "area_m2", "num_predios", "num_construcciones", "pisos_prom", "densidad_predial"]].head(10)

## 4. Eventos ITT (PLACEHOLDER - reemplazar con datos reales)

Esta celda es el UNICO punto donde se inyectan datos sinteticos. Cuando lleguen los eventos reales (hurtos, homicidios, rinas, lesionados, siniestralidad, SPA, VIF) y el censo arboreo, reemplazar el contenido por la carga/cruce espacial real a nivel de manzana (join por `IDMANZANA` o join espacial por coordenadas), manteniendo los mismos nombres de columna.

In [ ]:
# ADVERTENCIA: datos sinteticos de demostracion. NO usar para decisiones.
DATOS_SINTETICOS = True
rng = np.random.default_rng(42)
n = len(mz)

# Conteos de eventos por manzana (periodo actual T0 y anterior T-1 para tendencia)
for col in ["hurtos", "siniestros", "rinas", "spa", "lesionados", "homicidios", "vif"]:
    base = rng.integers(0, 12, size=n)
    mz[col + "_t0"] = base
    mz[col + "_t1"] = np.clip(base + rng.integers(-4, 5, size=n), 0, None)

# Censo arboreo (arboles por manzana) y NDVI promedio [0..1]
mz["arboles"] = rng.integers(0, 60, size=n)
mz["ndvi"] = (rng.uniform(0.1, 0.6, size=n)).round(3)

print("Eventos sinteticos generados (placeholder). Columnas de evento:")
print([c for c in mz.columns if c.endswith("_t0") or c in ("arboles", "ndvi")])

## 5. Indice ITT y clasificacion en 4 niveles

In [ ]:
def norm_inv(s):
    """Normaliza 0..1 e invierte: mas eventos negativos -> menor score."""
    s = s.astype(float)
    rng_ = s.max() - s.min()
    if rng_ == 0:
        return pd.Series(1.0, index=s.index)
    return 1.0 - (s - s.min()) / rng_

def norm_dir(s):
    """Normaliza 0..1 directo: mas es mejor (ej. arbolado)."""
    s = s.astype(float)
    rng_ = s.max() - s.min()
    if rng_ == 0:
        return pd.Series(1.0, index=s.index)
    return (s - s.min()) / rng_

# Dimensiones ITT (ejemplo 5 dim): seguridad, convivencia, movilidad, ambiente, consolidacion urbana
seguridad   = norm_inv(mz["hurtos_t0"] + mz["homicidios_t0"])
convivencia = norm_inv(mz["rinas_t0"] + mz["vif_t0"] + mz["spa_t0"])
movilidad   = norm_inv(mz["siniestros_t0"] + mz["lesionados_t0"])
ambiente    = (norm_dir(mz["arboles"]) + norm_dir(mz["ndvi"])) / 2.0
urbano      = norm_dir(mz["pisos_prom"]) 

mz["itt"] = ((seguridad + convivencia + movilidad + ambiente + urbano) / 5.0 * 100).round(1)

def nivel(itt):
    if itt < 40:  return "Activacion"
    if itt < 60:  return "Consolidacion"
    if itt < 80:  return "Transformacion"
    return "Escala"

mz["nivel"] = mz["itt"].map(nivel)
mz["color_nivel"] = mz["nivel"].map(NIVEL_COLORS)

print(mz["nivel"].value_counts().to_string())
print("\nITT min/med/max:", mz["itt"].min(), mz["itt"].median(), mz["itt"].max())

## 6. Exportar artefactos procesados (data_processed_geo/)

In [ ]:
mz_web = mz.to_crs(CRS_WEB)

geojson_path = os.path.join(OUT_DIR, "manzanas_itt.geojson")
parquet_path = os.path.join(OUT_DIR, "manzanas_itt.parquet")

mz_web.to_file(geojson_path, driver="GeoJSON")
try:
    mz_web.to_parquet(parquet_path)
    print("Parquet:", parquet_path)
except Exception as e:
    print("Parquet no generado (requiere pyarrow):", e)

print("GeoJSON:", geojson_path)
print("Manzanas exportadas:", len(mz_web))

## 7. Popups enriquecidos (HTML/CSS + mini-graficas SVG)

In [ ]:
def svg_barras(valores, colores, w=180, h=70):
    """Mini grafico de barras en SVG inline (sin dependencias JS en el popup)."""
    vmax = max(valores) if max(valores) > 0 else 1
    bw = w / (len(valores) * 1.5)
    gap = bw * 0.5
    barras = []
    for i, (v, c) in enumerate(zip(valores, colores)):
        bh = (v / vmax) * (h - 18)
        x = gap + i * (bw + gap)
        y = h - bh - 12
        barras.append(
            f'<rect x="{x:.1f}" y="{y:.1f}" width="{bw:.1f}" height="{bh:.1f}" '
            f'fill="{c}" stroke="{OKI_GRIS}" stroke-width="0.5" rx="1"/>'
            f'<text x="{x + bw/2:.1f}" y="{h-2}" font-size="8" fill="{OKI_GRIS}" '
            f'text-anchor="middle">{v}</text>'
        )
    return f'<svg width="{w}" height="{h}" style="background:{BG};border-radius:4px">' + "".join(barras) + "</svg>"

def pct_var(t0, t1):
    if t1 == 0:
        return "s/d" if t0 == 0 else "+100%"
    return f"{(t0 - t1) / t1 * 100:+.0f}%"

def popup_html(r):
    eventos = ["hurtos", "siniestros", "rinas", "spa", "lesionados", "homicidios", "vif"]
    labels = ["Hurtos", "Siniestros", "Rinas", "SPA", "Lesion.", "Homic.", "VIF"]
    vals = [int(r[e + "_t0"]) for e in eventos]
    cols = [EVENTO_COLORS[e] for e in eventos]
    svg = svg_barras(vals, cols)
    filas = "".join(
        f'<tr><td style="padding:1px 6px">{lab}</td>'
        f'<td style="text-align:right;padding:1px 6px"><b>{r[e+"_t0"]}</b></td>'
        f'<td style="text-align:right;padding:1px 6px;color:{OKI_GRIS}">{pct_var(r[e+"_t0"], r[e+"_t1"])}</td></tr>'
        for e, lab in zip(eventos, labels)
    )
    nivel_c = NIVEL_COLORS[r["nivel"]]
    aviso = ('<div style="color:#b00;font-size:9px;margin-top:4px">Datos de evento SINTETICOS (demo)</div>'
             if DATOS_SINTETICOS else "")
    return (
        f'<div style="font-family:system-ui,Segoe UI,sans-serif;font-size:12px;'
        f'background:{BG};padding:8px;border-radius:6px;min-width:230px">'
        f'<div style="font-size:13px;font-weight:700;color:{OKI_GRIS}">Manzana {r["MANZANA"]} '
        f'<span style="font-weight:400;color:#777">({r["IDMANZANA"]})</span></div>'
        f'<div style="color:#555">Comuna {r["COMUNA"]} - Barrio {r["BARRIO"]}</div>'
        f'<div style="margin:4px 0"><span style="background:{nivel_c};color:#fff;'
        f'padding:2px 8px;border-radius:999px;font-size:11px;font-weight:700">'
        f'{r["nivel"]} - ITT {r["itt"]}</span></div>'
        f'<div style="color:#555;font-size:11px">Area: {r["area_m2"]/10000:.2f} ha - '
        f'Predios: {r["num_predios"]} - Pisos prom: {r["pisos_prom"]}</div>'
        f'<div style="margin-top:6px;font-size:10px;color:{OKI_GRIS};font-weight:700">'
        f'Eventos T0 por tipo</div>{svg}'
        f'<table style="font-size:11px;border-collapse:collapse;margin-top:4px;width:100%">'
        f'<tr style="color:{OKI_GRIS};border-bottom:1px solid #ccc">'
        f'<td style="padding:1px 6px">Evento</td><td style="text-align:right;padding:1px 6px">T0</td>'
        f'<td style="text-align:right;padding:1px 6px">vs T-1</td></tr>{filas}</table>{aviso}</div>'
    )

print("Funciones de popup listas. Ejemplo de longitud HTML:", len(popup_html(mz.iloc[0])), "chars")

## 8. Construccion del mapa interactivo (Folium)

In [ ]:
import folium
from folium.plugins import HeatMap

centro = [mz_web.geometry.union_all().centroid.y, mz_web.geometry.union_all().centroid.x]
m = folium.Map(location=centro, zoom_start=16, tiles=None, control_scale=True)

# 4 capas base conmutables (OSM activa por defecto)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Esri Satelite", show=False).add_to(m)
folium.TileLayer(
    tiles="https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}",
    attr="Google", name="Google Streets", show=True).add_to(m)
folium.TileLayer(
    tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
    attr="Google", name="Google Satelite", show=False).add_to(m)

# Capa 1: contorno territorial (activa al inicio)
contorno = folium.FeatureGroup(name="Contorno territorial", show=True)
folium.GeoJson(
    mz_web.to_json(),
    style_function=lambda f: {"color": OKI_GRIS, "weight": 1.2, "fill": False},
).add_to(contorno)
contorno.add_to(m)

# Capa 2: coropleta por nivel ITT (apagada al inicio)
coropleta = folium.FeatureGroup(name="Coropleta ITT (nivel)", show=False)
for _, r in mz_web.iterrows():
    folium.GeoJson(
        r["geometry"].__geo_interface__,
        style_function=lambda f, c=NIVEL_PASTEL[r["nivel"]], b=NIVEL_COLORS[r["nivel"]]: {
            "fillColor": c, "color": b, "weight": 1.5, "fillOpacity": 0.7},
        popup=folium.Popup(popup_html(r), max_width=280),
        tooltip=f'Manzana {r["MANZANA"]} - {r["nivel"]}',
    ).add_to(coropleta)
coropleta.add_to(m)

# Capa del poligono de la zona de intervencion (Ciudad Paraiso), borde dorado.
if POLIGONO_ZONA is not None:
    zona_fg = folium.FeatureGroup(name="Poligono Ciudad Paraiso", show=True)
    folium.GeoJson(
        POLIGONO_ZONA.to_json(),
        style_function=lambda f: {"color": "#8A6D1A", "weight": 5,
                                  "dashArray": "10 6", "fill": False},
        tooltip="Zona de intervencion: Ciudad Paraiso",
    ).add_to(zona_fg)
    zona_fg.add_to(m)
print("Mapa base + contorno + coropleta construidos.")

In [ ]:
# Capa 3: coropleta de densidad de hurtos por manzana (gradiente azul por cantidad).
# Representa mejor el territorio que un punto en el centroide. Con datos reales,
# usar el conteo real de hurtos por manzana (o un join espacial de los eventos).
def _rgb_hex(h):
    h = h.lstrip("#")
    return tuple(int(h[k:k + 2], 16) for k in (0, 2, 4))
def _grad_azul(pct):
    r, g, b = _rgb_hex(EVENTO_COLORS["hurtos"])
    t = max(0.0, min(1.0, pct))
    return "#%02X%02X%02X" % (int(255 + (r - 255) * t), int(255 + (g - 255) * t), int(255 + (b - 255) * t))
# --- diagrama de torta SVG: composicion de delitos de la manzana ---
DELITOS = ["hurtos", "homicidios", "rinas", "lesionados", "siniestros", "spa"]
DELITO_LABEL = {"hurtos": "Hurtos", "homicidios": "Homicidios", "rinas": "Rinas",
                "lesionados": "Lesionados", "siniestros": "Siniestros", "spa": "SPA"}
def svg_torta(valores, colores, size=120):
    total = float(sum(valores))
    cx = cy = size / 2.0
    rad = size / 2.0 - 4
    if total <= 0:
        return ('<svg width="%d" height="%d"><circle cx="%.1f" cy="%.1f" r="%.1f" '
                'fill="#E8E8E8" stroke="%s" stroke-width="0.5"/>'
                '<text x="%.1f" y="%.1f" font-size="9" fill="%s" text-anchor="middle">sin datos</text>'
                '</svg>') % (size, size, cx, cy, rad, OKI_GRIS, cx, cy, OKI_GRIS)
    partes = []
    ang0 = -90.0
    for v, c in zip(valores, colores):
        if v <= 0:
            continue
        frac = v / total
        ang1 = ang0 + frac * 360.0
        x0 = cx + rad * math.cos(math.radians(ang0))
        y0 = cy + rad * math.sin(math.radians(ang0))
        x1 = cx + rad * math.cos(math.radians(ang1))
        y1 = cy + rad * math.sin(math.radians(ang1))
        large = 1 if (ang1 - ang0) > 180 else 0
        if frac >= 0.999:
            partes.append('<circle cx="%.1f" cy="%.1f" r="%.1f" fill="%s" stroke="#fff" stroke-width="0.8"/>'
                          % (cx, cy, rad, c))
        else:
            partes.append('<path d="M%.1f,%.1f L%.1f,%.1f A%.1f,%.1f 0 %d,1 %.1f,%.1f Z" '
                          'fill="%s" stroke="#fff" stroke-width="0.8"/>'
                          % (cx, cy, x0, y0, rad, rad, large, x1, y1, c))
        ang0 = ang1
    return '<svg width="%d" height="%d" style="background:%s;border-radius:6px">%s</svg>' % (
        size, size, BG, "".join(partes))
def popup_hurtos_html(r):
    vals = [int(r[d + "_t0"]) for d in DELITOS]
    cols = [EVENTO_COLORS[d] for d in DELITOS]
    torta = svg_torta(vals, cols)
    total = sum(vals)
    leyenda = "".join(
        '<div style="display:flex;align-items:center;gap:5px;margin:1px 0">'
        '<i style="width:10px;height:10px;background:%s;display:inline-block;border-radius:2px;'
        'border:0.5px solid %s"></i><span style="font-size:10px">%s</span></div>'
        % (EVENTO_COLORS[d], OKI_GRIS, DELITO_LABEL[d]) for d in DELITOS
    )
    filas = "".join(
        '<tr><td style="padding:1px 6px">%s</td>'
        '<td style="text-align:right;padding:1px 6px"><b>%d</b></td>'
        '<td style="text-align:right;padding:1px 6px">%s</td>'
        '<td style="text-align:right;padding:1px 6px;color:%s">%s</td></tr>'
        % (DELITO_LABEL[d], int(r[d + "_t0"]),
           ("%.0f%%" % (100.0 * int(r[d + "_t0"]) / total)) if total else "-",
           OKI_GRIS, pct_var(int(r[d + "_t0"]), int(r[d + "_t1"])))
        for d in DELITOS
    )
    return (
        '<div style="font-family:system-ui,Segoe UI,sans-serif;font-size:12px;'
        'background:%s;padding:8px;border-radius:6px;min-width:260px">'
        '<div style="font-size:13px;font-weight:700;color:%s">Delitos - Manzana %s '
        '<span style="font-weight:400;color:#777">(%s)</span></div>'
        '<div style="color:#555">Comuna %s - Barrio %s - Total T0: <b>%d</b></div>'
        '<div style="display:flex;gap:10px;align-items:center;margin-top:6px">'
        '%s<div>%s</div></div>'
        '<table style="font-size:11px;border-collapse:collapse;margin-top:6px;width:100%%">'
        '<tr style="color:%s;border-bottom:1px solid #ccc">'
        '<td style="padding:1px 6px">Delito</td>'
        '<td style="text-align:right;padding:1px 6px">T0</td>'
        '<td style="text-align:right;padding:1px 6px">%% del total</td>'
        '<td style="text-align:right;padding:1px 6px">vs T-1</td></tr>%s</table>'
        '<div style="font-size:9px;color:#b00;margin-top:4px">Datos de evento SINTETICOS (demo)</div>'
        '</div>'
    ) % (BG, OKI_GRIS, r["MANZANA"], r["IDMANZANA"] if "IDMANZANA" in r else "",
         r["COMUNA"], r["BARRIO"], total, torta, leyenda, OKI_GRIS, filas)
_hmax = float(mz_web["hurtos_t0"].max()) or 1.0
hurtos_cp = folium.FeatureGroup(name="Hurtos (densidad por manzana)", show=False)
for _, r in mz_web.iterrows():
    val = int(r["hurtos_t0"])
    fill = _grad_azul(val / _hmax) if val > 0 else "#E8E8E8"
    folium.GeoJson(
        r["geometry"].__geo_interface__,
        style_function=(lambda f, c=fill: {"fillColor": c, "color": OKI_GRIS,
                                            "weight": 1.0, "fillOpacity": 0.8}),
        popup=folium.Popup(popup_hurtos_html(r), max_width=320),
        tooltip=f'Hurtos T0: {val} (manzana {r["MANZANA"]})',
    ).add_to(hurtos_cp)
hurtos_cp.add_to(m)

# Capa 4: VIF como HeatMap (privacidad: NUNCA punto individual)
vif = folium.FeatureGroup(name="VIF (mapa de calor)", show=False)
pts = []
for _, r in mz_web.iterrows():
    cen = r["geometry"].centroid
    peso = float(r["vif_t0"])
    if peso > 0:
        pts.append([cen.y, cen.x, peso])
if pts:
    HeatMap(pts, radius=28, blur=22, min_opacity=0.3).add_to(vif)
vif.add_to(m)

print("Capas de densidad de hurtos y HeatMap VIF agregadas. Puntos VIF:", len(pts))

## 9. Leyenda, control de capas y exportacion del HTML

In [ ]:
# Leyenda de niveles ITT (HTML flotante)
items = "".join(
    f'<div style="display:flex;align-items:center;gap:6px;margin:2px 0">'
    f'<i style="width:14px;height:14px;background:{c};border:1px solid {OKI_GRIS};'
    f'display:inline-block;border-radius:3px"></i><span>{k}</span></div>'
    for k, c in NIVEL_COLORS.items()
)
aviso = ('<div style="color:#b00;font-size:10px;margin-top:4px">Eventos: datos sinteticos (demo)</div>'
         if DATOS_SINTETICOS else "")
legend = (
    f'<div style="position:fixed;bottom:24px;left:12px;z-index:9999;background:{BG};'
    f'padding:10px 12px;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,.25);'
    f'font-family:system-ui,Segoe UI,sans-serif;font-size:12px;color:{OKI_GRIS}">'
    f'<div style="font-weight:700;margin-bottom:4px">Nivel ITT</div>{items}{aviso}</div>'
)
m.get_root().html.add_child(folium.Element(legend))

folium.LayerControl(collapsed=False, position="topright").add_to(m)

html_path = os.path.join(OUT_DIR, "mapa_geoanalitica_itt.html")
m.save(html_path)
print("Mapa exportado:", html_path)
m

## 10. Resumen analitico de hallazgos territoriales

In [ ]:
resumen = {
    "manzanas_analizadas": int(len(mz)),
    "predios_totales": int(mz["num_predios"].sum()),
    "construcciones_totales": int(mz["num_construcciones"].sum()),
    "itt_promedio": float(mz["itt"].mean().round(1)),
    "por_nivel": mz["nivel"].value_counts().to_dict(),
    "datos_eventos_sinteticos": DATOS_SINTETICOS,
}
print(json.dumps(resumen, ensure_ascii=False, indent=2))

print("\nHallazgos (sobre indicadores catastrales reales):")
top_den = mz.sort_values("densidad_predial", ascending=False).head(3)
for _, r in top_den.iterrows():
    print(f"  - Manzana {r['MANZANA']} ({r['IDMANZANA']}): {r['densidad_predial']} predios/ha, "
          f"{r['num_predios']} predios, pisos prom {r['pisos_prom']}")
print("\nNota: la clasificacion ITT depende de los eventos; con datos sinteticos los niveles")
print("son ilustrativos. Los indicadores catastrales (predios, area, pisos, densidad) SI son reales.")

## 11. Avance de caracterizacion por manzana (datos reales)
Nueva tematica sobre datos reales de la encuesta (`data/*.xlsx`). Se reutiliza la logica de geolocalizacion (GPS por prioridad Ficha > Hogares > Salud > Empresas, y direccion declarada sobre grilla calle/carrera) y de clasificacion de avance por eje del notebook de visitas. El avance se separa en 4 ejes tematicos: **Ficha, Hogares, Salud y Empresas**. En el mapa solo se publican agregados por manzana (nunca puntos de personas).

### 11a. Carga de los Excel de la encuesta

In [ ]:
# SECCION_11_MARK
# 11a. Carga de los 5 Excel de la encuesta (datos reales en data/).
# Lectura directa de los archivos locales (sin modelo S3). El archivo de
# caracterizacion lleva acento en el nombre: se localiza por glob con 'caracteriz'.
import glob, io, math, re, unicodedata

def _resolver_data_dir():
    cands = [os.path.join("..", "data"), os.path.join("data"), "."]
    env = os.environ.get("DATA_DIR")
    if env:
        cands = [env] + cands
    for c in cands:
        if c and os.path.isfile(os.path.join(c, "ListadoCiudadanosCompleto.xlsx")):
            return os.path.abspath(c)
    raise FileNotFoundError("No se encontro data/ con los Excel de la encuesta.")

DATA_DIR = _resolver_data_dir()

def _primer_match(token):
    fs = glob.glob(os.path.join(DATA_DIR, "*" + token + "*"))
    if not fs:
        raise FileNotFoundError("No se encontro Excel con token '" + token + "'")
    return fs[0]

# Lectura como texto (dtype=str) para preservar documentos y coordenadas tal cual.
mae = pd.read_excel(_primer_match("ListadoCiudadanosCompleto"), sheet_name="Ciudadanos", dtype=str)
hog = pd.read_excel(_primer_match("ListadoHogaresIntegrantes"), sheet_name="Hogares", dtype=str)
fic = pd.read_excel(_primer_match("caracteriz"), sheet_name=0, dtype=str)
prog = pd.read_excel(_primer_match("respuestas_programas"), sheet_name=None, dtype=str)

# Nombres de columnas clave (con acentos tal como vienen en los Excel).
COL_DOC_MAE = "Numero de documento" if "Numero de documento" in mae.columns else "N\u00famero de documento"
COL_DOC_FIC = "Numero de identificacion" if "Numero de identificacion" in fic.columns else "N\u00famero de identificaci\u00f3n"

print("DATA_DIR:", DATA_DIR)
print("Ciudadanos (padron):", len(mae), "| docs unicos:", mae[COL_DOC_MAE].nunique())
print("Hogares integrantes:", len(hog))
print("Ficha caracterizacion:", len(fic))
for k, v in prog.items():
    print("  Programa", k.replace("Programa ", ""), "->", len(v), "filas")


### 11b. Geolocalizacion y asignacion a manzana

In [ ]:
# 11b. Geolocalizacion de cada ciudadano y asignacion a manzana catastral.
# Logica portada TAL CUAL del notebook de visitas (Mapa_Visitas_Colab_v1):
#   GPS por prioridad de fuente Ficha > Hogares > Salud > Empresas, y, si no hay
#   GPS, estimacion de la direccion declarada sobre una grilla calle/carrera
#   ajustada por minimos cuadrados con los puntos GPS de ficha. Luego punto-en-
#   poligono sobre las manzanas catastrales reproyectadas a WGS84 (EPSG:4326).

# --- utilidades de texto (portadas de etl_handler.py) ---
def _s(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    x = str(x).strip()
    return x or None

# --- parseo de coordenadas y direcciones (portado tal cual) ---
PAT = re.compile(
    r"(?P<t>avenida carrera|calle|cll|cl|carrera|cra|cr|kr|kra)\.?\s*"
    r"(?P<m>\d{1,3})\s*(?P<ma>[a-zA-Z])?\s*(?:#|no\.?|n\u00b0|n\u00famero|numero|nro\.?)?"
    r"\s*[-\s]*(?P<c>\d{1,3})\s*(?P<ca>[a-zA-Z])?\s*(?:#|-|\u2013|\s)\s*(?P<h>\d{1,3})\b",
    re.I,
)
CALI_BBOX = (3.30, 3.60, -76.65, -76.40)
ZONA_BBOX = (3.425, 3.465, -76.545, -76.515)

def parse_addr(txt):
    """-> (norte, este, tipo) en indices de calle/carrera, o None."""
    if not isinstance(txt, str):
        return None
    mt = PAT.search(txt.split(",")[0])
    if not mt:
        return None
    t = "calle" if mt["t"].lower() in ("calle", "cll", "cl") else "carrera"
    main = int(mt["m"]) + (0.5 if mt["ma"] else 0)
    cross = int(mt["c"]) + (0.5 if mt["ca"] else 0)
    h = int(mt["h"])
    if h > 200:
        h //= 10
    return (cross + min(h, 99) / 100, main, t) if t == "carrera" else (main, cross + min(h, 99) / 100, t)

def parse_coord(v):
    v = _s(v)
    if not v:
        return None
    mt = re.findall(r"-?\d+\.\d+", v)
    if len(mt) < 2:
        return None
    la, lo = float(mt[0]), float(mt[1])
    return (la, lo) if (-90 <= la <= 90 and -180 <= lo <= 180) else None

def _hav(a, b):
    return math.hypot((a[0] - b[0]) * 111000,
                      (a[1] - b[1]) * 111000 * math.cos(math.radians(3.44)))

# --- construccion de ubicacion por persona (GPS por prioridad + direcciones) ---
EJES = ["ficha", "hogares", "salud", "empresas"]
FUENTES_GPS = [
    ("ficha", fic, COL_DOC_FIC),
    ("hogares", prog["Programa Hogares"], COL_DOC_FIC),
    ("salud", prog["Programa Salud"], COL_DOC_FIC),
    ("empresas", prog["Programa Empresas"], COL_DOC_FIC),
]

def build_ubicacion():
    gps, ubic_txt = {}, {}
    for nombre, df, kcol in FUENTES_GPS:
        if "Coordenadas" not in df.columns:
            continue
        d2 = df.drop_duplicates(kcol, keep="last")
        ucol = "Ubicacion" if "Ubicacion" in d2.columns else ("Ubicaci\u00f3n" if "Ubicaci\u00f3n" in d2.columns else None)
        for i, row in d2.iterrows():
            doc = row[kcol]
            cc = parse_coord(row.get("Coordenadas"))
            if cc:
                gps.setdefault(doc, []).append((nombre, cc))
            if ucol and _s(row.get(ucol)):
                ubic_txt.setdefault(doc, _s(row.get(ucol)))
    mae_i = mae.drop_duplicates(COL_DOC_MAE).set_index(COL_DOC_MAE)
    col_dir = "Direccion" if "Direccion" in mae.columns else "Direcci\u00f3n"
    col_comp = "Complemento de direccion" if "Complemento de direccion" in mae.columns else "Complemento de direcci\u00f3n"
    rows = []
    docs = list(mae_i.index)
    for doc in docs:
        lst = gps.get(doc, [])
        en_cali = [(n, c) for n, c in lst
                   if CALI_BBOX[0] <= c[0] <= CALI_BBOX[1] and CALI_BBOX[2] <= c[1] <= CALI_BBOX[3]]
        lat = lon = None
        fuente = None
        if en_cali:
            fuente, (lat, lon) = en_cali[0]
        pad = None
        if doc in mae_i.index:
            pad = " ".join(x for x in [_s(mae_i.at[doc, col_dir]) if col_dir in mae_i.columns else None,
                                       _s(mae_i.at[doc, col_comp]) if col_comp in mae_i.columns else None] if x) or None
        rows.append(dict(doc=doc, lat=lat, lon=lon, fuente=fuente,
                         dir_padron=pad, ubic_form=ubic_txt.get(doc),
                         declarada=bool(pad)))
    return pd.DataFrame(rows)

U = build_ubicacion()

def _addr_parse_row(r):
    for c in ("ubic_form", "dir_padron"):
        p = parse_addr(r[c])
        if p:
            return p
    return None

# --- ajuste de la grilla calle/carrera por minimos cuadrados (portado) ---
def fit_grid(U):
    U["p"] = U.apply(_addr_parse_row, axis=1)
    tr = U[U.p.notna() & U.lat.notna() & (U.fuente == "ficha") & U.declarada]
    if len(tr) < 30:
        return None, None, {"puntos": int(len(tr)), "aviso": "Muy pocos puntos para ajustar la grilla"}
    X = np.array([[1, p[0], p[1]] for p in tr.p])
    la, lo = tr.lat.values.astype(float), tr.lon.values.astype(float)
    mask = np.ones(len(tr), bool)
    dd = np.zeros(len(tr))
    for _ in range(5):
        cl = np.linalg.lstsq(X[mask], la[mask], rcond=None)[0]
        co = np.linalg.lstsq(X[mask], lo[mask], rcond=None)[0]
        dd = np.hypot((X @ cl - la) * 111000, (X @ co - lo) * 111000 * np.cos(np.radians(3.44)))
        mask = dd <= max(np.percentile(dd[mask], 90), 40)
    return cl, co, {"puntos_entrenamiento": int(len(tr)), "inliers": int(mask.sum()),
                    "error_mediano_m": round(float(np.median(dd[mask])), 1)}

def apply_grid(U, cl, co):
    U["ubic"] = np.where(U.lat.notna(), "GPS", "NINGUNA")
    if cl is None:
        return
    for i, r in U[U.lat.isna()].iterrows():
        p = r.p
        if p:
            la = float(cl @ np.array([1, p[0], p[1]]))
            lo = float(co @ np.array([1, p[0], p[1]]))
            if ZONA_BBOX[0] <= la <= ZONA_BBOX[1] and ZONA_BBOX[2] <= lo <= ZONA_BBOX[3]:
                U.loc[i, ["lat", "lon", "ubic"]] = [round(la, 6), round(lo, 6), "APROX"]

CL, CO, GRID_INFO = fit_grid(U)
apply_grid(U, CL, CO)

# --- asignacion a manzana por punto-en-poligono sobre manzanas en WGS84 ---
mz_wgs = mz.to_crs(CRS_WEB)[["IDMANZANA", "COMUNA", "BARRIO", "MANZANA", "geometry"]].copy()

def _pip(x, y, ring):
    c, n = False, len(ring)
    for i in range(n):
        x1, y1 = ring[i]
        x2, y2 = ring[(i + 1) % n]
        if (y1 > y) != (y2 > y) and x < (x2 - x1) * (y - y1) / (y2 - y1) + x1:
            c = not c
    return c

def _rings_de_geom(geom):
    gj = geom.__geo_interface__
    if gj["type"] == "Polygon":
        return [gj["coordinates"][0]]
    if gj["type"] == "MultiPolygon":
        return [poly[0] for poly in gj["coordinates"]]
    return []

MZ_RINGS = [(row["IDMANZANA"], _rings_de_geom(row["geometry"])) for _, row in mz_wgs.iterrows()]

# Distancia (m) de un punto al borde de un anillo (portada de etl_handler.py).
# Permite el "snap al borde": asignar a la manzana mas cercana los puntos que
# caen justo fuera del poligono (p. ej. sobre la calle) hasta SNAP_METROS.
SNAP_METROS = 25.0
def _dist_ring_m(x, y, ring):
    best = 1e18
    for i in range(len(ring)):
        x1, y1 = ring[i]
        x2, y2 = ring[(i + 1) % len(ring)]
        dx, dy = x2 - x1, y2 - y1
        t = max(0, min(1, ((x - x1) * dx + (y - y1) * dy) / (dx * dx + dy * dy or 1)))
        best = min(best, math.hypot((x - x1 - t * dx) * 111320 * 0.9984, (y - y1 - t * dy) * 110574))
    return best
def asignar_manzana(lat, lon):
    """Devuelve (IDMANZANA, calidad, dist_m). calidad: DENTRO / APROXIMADA / None."""
    if lat is None or (isinstance(lat, float) and np.isnan(lat)):
        return (None, None, None)
    for idm, rings in MZ_RINGS:
        if any(_pip(lon, lat, ring) for ring in rings):
            return (idm, "DENTRO", 0.0)
    mejor_id, mejor_d = None, 1e18
    for idm, rings in MZ_RINGS:
        d = min((_dist_ring_m(lon, lat, ring) for ring in rings), default=1e18)
        if d < mejor_d:
            mejor_id, mejor_d = idm, d
    if mejor_id is not None and mejor_d <= SNAP_METROS:
        return (mejor_id, "APROXIMADA", round(mejor_d, 1))
    return (None, None, round(mejor_d, 1) if mejor_id is not None else None)
_asig = [asignar_manzana(r.lat, r.lon) for r in U.itertuples()]
U["IDMANZANA"] = [a[0] for a in _asig]
U["mz_calidad"] = [a[1] for a in _asig]
U["mz_dist_m"] = [a[2] for a in _asig]
n_ubic = int(U["lat"].notna().sum())
n_gps = int((U["ubic"] == "GPS").sum())
n_aprox = int((U["ubic"] == "APROX").sum())
n_mz = int(U["IDMANZANA"].notna().sum())
n_dentro = int((U["mz_calidad"] == "DENTRO").sum())
n_borde = int((U["mz_calidad"] == "APROXIMADA").sum())
print("Grilla:", GRID_INFO)
print("Ciudadanos totales:", len(U))
print("Ubicados (lat/lon):", n_ubic, "| GPS:", n_gps, "| APROX (grilla):", n_aprox)
print("Con manzana asignada:", n_mz, "| dentro del poligono:", n_dentro,
      "| por snap al borde (<= %dm):" % SNAP_METROS, n_borde)


### 11c. Estado por eje y agregacion por manzana

In [ ]:
# 11c. Estado por eje (Ficha, Hogares, Salud, Empresas) y agregacion por manzana.
# Estado por eje de cada ciudadano (regla pstate del notebook de visitas):
#   '% de avance' == 100 -> COMPLETO ; == 0 -> VACIO ; resto -> PARCIAL.
# El eje Ficha es presencia en la caracterizacion: aparece -> COMPLETO, si no -> VACIO.

def pstate(df, kcol):
    """Serie doc -> estado (COMPLETO/PARCIAL/VACIO) segun '% de avance'."""
    d = df.copy()
    d["av"] = pd.to_numeric(d["% de avance"], errors="coerce")
    d["doc"] = d[kcol]
    d["st"] = np.where(d.av == 100, "COMPLETO", np.where(d.av == 0, "VACIO", "PARCIAL"))
    return d.drop_duplicates("doc", keep="last").set_index("doc")["st"]

docs_fic = set(fic[COL_DOC_FIC].dropna())
H = pstate(prog["Programa Hogares"], COL_DOC_FIC)
S = pstate(prog["Programa Salud"], COL_DOC_FIC)
E = pstate(prog["Programa Empresas"], COL_DOC_FIC)

# Tabla por ciudadano con el estado de los 4 ejes y su manzana (clave: IDMANZANA).
estado = U[["doc", "IDMANZANA", "mz_calidad"]].copy()
estado["ficha"] = np.where(estado.doc.isin(docs_fic), "COMPLETO", "VACIO")
estado["hogares"] = estado.doc.map(H).fillna("VACIO")
estado["salud"] = estado.doc.map(S).fillna("VACIO")
estado["empresas"] = estado.doc.map(E).fillna("VACIO")

# Puntaje de avance por eje por ciudadano: COMPLETO=1.0, PARCIAL=0.5, VACIO=0.0.
PESO = {"COMPLETO": 1.0, "PARCIAL": 0.5, "VACIO": 0.0}
for eje in EJES:
    estado[eje + "_sc"] = estado[eje].map(PESO)

# Solo ciudadanos con manzana asignada entran a la agregacion espacial.
con_mz = estado[estado["IDMANZANA"].notna()].copy()

def agregar_por_manzana(g):
    n = len(g)
    out = {"n_ciudadanos": n}
    out["n_aprox"] = int((g["mz_calidad"] == "APROXIMADA").sum())
    out["pct_aprox"] = round(100.0 * out["n_aprox"] / n, 1) if n else 0.0
    for eje in EJES:
        # % de avance del eje en la manzana = promedio del puntaje * 100.
        out["pct_" + eje] = round(float(g[eje + "_sc"].mean() * 100), 1) if n else 0.0
        out["n_" + eje + "_completo"] = int((g[eje] == "COMPLETO").sum())
        out["n_" + eje + "_parcial"] = int((g[eje] == "PARCIAL").sum())
    # Avance global de la manzana = promedio de los 4 ejes.
    out["pct_global"] = round(float(np.mean([out["pct_" + e] for e in EJES])), 1)
    return pd.Series(out)

agg = con_mz.groupby("IDMANZANA").apply(agregar_por_manzana).reset_index()

# GeoDataFrame de manzanas (en WGS84) con los agregados. Manzanas sin ciudadanos
# quedan con 0 y n_ciudadanos=0 (se grafican como "sin datos").
cols_pct = ["pct_ficha", "pct_hogares", "pct_salud", "pct_empresas", "pct_global"]
mz_av = mz_wgs.merge(agg, on="IDMANZANA", how="left")
for c in cols_pct:
    mz_av[c] = mz_av[c].fillna(0.0)
mz_av["n_ciudadanos"] = mz_av["n_ciudadanos"].fillna(0).astype(int)
mz_av["n_aprox"] = mz_av["n_aprox"].fillna(0).astype(int)
mz_av["pct_aprox"] = mz_av["pct_aprox"].fillna(0.0)
for eje in EJES:
    for suf in ("_completo", "_parcial"):
        col = "n_" + eje + suf
        mz_av[col] = mz_av[col].fillna(0).astype(int)

print("Ciudadanos con manzana:", len(con_mz), "en", con_mz["IDMANZANA"].nunique(), "manzanas")
print("Manzanas con al menos 1 ciudadano:", int((mz_av["n_ciudadanos"] > 0).sum()), "de", len(mz_av))
mz_av[["IDMANZANA", "n_ciudadanos"] + cols_pct].sort_values("n_ciudadanos", ascending=False).head(10)


### 11d. Exportacion de agregados (sin PII)

In [ ]:
# 11d. Exportar agregados por manzana (SIN PII) a data_processed_geo/.
# Solo se exportan conteos y porcentajes por manzana; nunca puntos de personas.
cols_exportar = ["IDMANZANA", "COMUNA", "BARRIO", "MANZANA", "n_ciudadanos",
                 "pct_ficha", "pct_hogares", "pct_salud", "pct_empresas", "pct_global"]
for eje in EJES:
    cols_exportar += ["n_" + eje + "_completo", "n_" + eje + "_parcial"]
mz_av_out = mz_av[cols_exportar + ["geometry"]].copy()

geojson_av = os.path.join(OUT_DIR, "avance_caracterizacion_manzana.geojson")
parquet_av = os.path.join(OUT_DIR, "avance_caracterizacion_manzana.parquet")
mz_av_out.to_file(geojson_av, driver="GeoJSON")
try:
    mz_av_out.to_parquet(parquet_av)
    print("Parquet:", parquet_av)
except Exception as e:
    print("Parquet no generado (requiere pyarrow):", e)
print("GeoJSON:", geojson_av)
print("Manzanas exportadas:", len(mz_av_out), "(solo agregados, sin PII)")


### 11e. Mapa interactivo del avance por eje

In [ ]:
# 11e. Mapa Folium del avance de caracterizacion por manzana.
# Estructura del mapa ITT: 4 capas base (OSM / Esri / Google Streets / Google
# Satelite) + contorno. Encima, una coropleta conmutable por CADA eje tematico
# (gradiente del color del eje segun el % de avance) y una coropleta de avance
# global. LayerControl con todo apagado salvo el contorno territorial.
import folium

# Color base por eje (segun el requerimiento).
EJE_COLOR = {
    "ficha": OKI_VERDE,      # verde
    "hogares": OKI_NARANJA,  # naranja
    "salud": OKI_AZUL_C,     # azul cielo
    "empresas": OKI_AZUL,    # azul
}
EJE_LABEL = {"ficha": "Ficha", "hogares": "Hogares", "salud": "Salud", "empresas": "Empresas"}
GLOBAL_COLOR = OKI_MORADO

def _hex_a_rgb(h):
    h = h.lstrip("#")
    return tuple(int(h[i:i + 2], 16) for i in (0, 2, 4))

def gradiente(color_hex, pct):
    """Gradiente del color del eje: blanco (0%) -> color saturado (100%)."""
    r, g, b = _hex_a_rgb(color_hex)
    t = max(0.0, min(1.0, pct / 100.0))
    rr = int(255 + (r - 255) * t)
    gg = int(255 + (g - 255) * t)
    bb = int(255 + (b - 255) * t)
    return "#%02X%02X%02X" % (rr, gg, bb)

# --- popup enriquecido por manzana: barras SVG comparando los 4 ejes + tabla ---
def svg_barras_ejes(pcts, colores, w=200, h=80):
    vmax = 100.0
    bw = w / (len(pcts) * 1.5)
    gap = bw * 0.5
    barras = []
    for i, (v, c) in enumerate(zip(pcts, colores)):
        bh = (v / vmax) * (h - 20)
        x = gap + i * (bw + gap)
        y = h - bh - 14
        barras.append(
            '<rect x="%.1f" y="%.1f" width="%.1f" height="%.1f" fill="%s" '
            'stroke="%s" stroke-width="0.5" rx="1"/>'
            '<text x="%.1f" y="%s" font-size="8" fill="%s" text-anchor="middle">%.0f</text>'
            % (x, y, bw, bh, c, OKI_GRIS, x + bw / 2, h - 2, OKI_GRIS, v)
        )
    return ('<svg width="%d" height="%d" style="background:%s;border-radius:4px">'
            % (w, h, BG)) + "".join(barras) + "</svg>"

def popup_av_html(r):
    pcts = [r["pct_" + e] for e in EJES]
    cols = [EJE_COLOR[e] for e in EJES]
    svg = svg_barras_ejes(pcts, cols)
    filas = "".join(
        '<tr><td style="padding:1px 6px">%s</td>'
        '<td style="text-align:right;padding:1px 6px"><b>%.0f%%</b></td>'
        '<td style="text-align:right;padding:1px 6px;color:%s">%d / %d</td></tr>'
        % (EJE_LABEL[e], r["pct_" + e], OKI_GRIS,
           int(r["n_" + e + "_completo"]), int(r["n_ciudadanos"]))
        for e in EJES
    )
    return (
        '<div style="font-family:system-ui,Segoe UI,sans-serif;font-size:12px;'
        'background:%s;padding:8px;border-radius:6px;min-width:240px">'
        '<div style="font-size:13px;font-weight:700;color:%s">Manzana %s '
        '<span style="font-weight:400;color:#777">(%s)</span></div>'
        '<div style="color:#555">Comuna %s - Barrio %s</div>'
        '<div style="margin:4px 0"><span style="background:%s;color:#fff;'
        'padding:2px 8px;border-radius:999px;font-size:11px;font-weight:700">'
        'Avance global %.0f%%</span> '
        '<span style="color:#555;font-size:11px">%d ciudadanos ubicados</span></div>'
        '<div style="margin-top:6px;font-size:10px;color:%s;font-weight:700">'
        '%% de avance por eje</div>%s'
        '<table style="font-size:11px;border-collapse:collapse;margin-top:4px;width:100%%">'
        '<tr style="color:%s;border-bottom:1px solid #ccc">'
        '<td style="padding:1px 6px">Eje</td>'
        '<td style="text-align:right;padding:1px 6px">%% avance</td>'
        '<td style="text-align:right;padding:1px 6px">completos</td></tr>'
        '%s</table>'
        '<div style="font-size:10px;color:#666;margin-top:4px">Ubicacion: '
        '%d dentro del poligono, %d aprox. al borde (&lt;=25 m)</div></div>'
    ) % (BG, OKI_GRIS, r["MANZANA"], r["IDMANZANA"], r["COMUNA"], r["BARRIO"],
         GLOBAL_COLOR, r["pct_global"], int(r["n_ciudadanos"]),
         OKI_GRIS, svg, OKI_GRIS, filas,
         int(r["n_ciudadanos"]) - int(r["n_aprox"]), int(r["n_aprox"]))

centro_av = [mz_wgs.geometry.union_all().centroid.y, mz_wgs.geometry.union_all().centroid.x]
m_av = folium.Map(location=centro_av, zoom_start=16, tiles=None, control_scale=True)

# 4 capas base conmutables (igual que el mapa ITT).
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Esri Satelite", show=False).add_to(m_av)
folium.TileLayer(
    tiles="https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}",
    attr="Google", name="Google Streets", show=True).add_to(m_av)
folium.TileLayer(
    tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
    attr="Google", name="Google Satelite", show=False).add_to(m_av)

# Contorno territorial (unica capa activa al inicio).
contorno_av = folium.FeatureGroup(name="Contorno territorial", show=True)
folium.GeoJson(
    mz_wgs.to_json(),
    style_function=lambda f: {"color": OKI_GRIS, "weight": 1.2, "fill": False},
).add_to(contorno_av)
contorno_av.add_to(m_av)

# Una coropleta por eje (gradiente del color del eje por % de avance).
def add_coropleta(nombre, campo_pct, color_hex):
    fg = folium.FeatureGroup(name=nombre, show=False)
    for _, r in mz_av.iterrows():
        pct = r[campo_pct]
        fill = gradiente(color_hex, pct) if r["n_ciudadanos"] > 0 else "#E8E8E8"
        # Borde punteado si la mayoria de ciudadanos de la manzana se asigno por
        # snap al borde (ubicacion aproximada); borde solido si predomina DENTRO.
        dash = "4 3" if (r["n_ciudadanos"] > 0 and r["pct_aprox"] > 50) else None
        folium.GeoJson(
            r["geometry"].__geo_interface__,
            style_function=(lambda f, c=fill, d=dash: {"fillColor": c, "color": OKI_GRIS,
                                                "weight": 1.0, "fillOpacity": 0.78,
                                                "dashArray": d}),
            popup=folium.Popup(popup_av_html(r), max_width=300),
            tooltip="Manzana %s - %s %.0f%%" % (r["MANZANA"], campo_pct.replace("pct_", ""), pct),
        ).add_to(fg)
    fg.add_to(m_av)

for eje in EJES:
    add_coropleta("Avance " + EJE_LABEL[eje], "pct_" + eje, EJE_COLOR[eje])
add_coropleta("Avance global", "pct_global", GLOBAL_COLOR)

# Leyenda flotante (gradiente y colores por eje).
leyenda_ejes = "".join(
    '<div style="display:flex;align-items:center;gap:6px;margin:2px 0">'
    '<i style="width:14px;height:14px;background:%s;border:1px solid %s;'
    'display:inline-block;border-radius:3px"></i><span>%s</span></div>'
    % (EJE_COLOR[e], OKI_GRIS, EJE_LABEL[e]) for e in EJES
)
leyenda_ejes += (
    '<div style="display:flex;align-items:center;gap:6px;margin:2px 0">'
    '<i style="width:14px;height:14px;background:%s;border:1px solid %s;'
    'display:inline-block;border-radius:3px"></i><span>Global</span></div>'
    % (GLOBAL_COLOR, OKI_GRIS)
)
legend_av = (
    '<div style="position:fixed;bottom:24px;left:12px;z-index:9999;background:%s;'
    'padding:10px 12px;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,.25);'
    'font-family:system-ui,Segoe UI,sans-serif;font-size:12px;color:%s">'
    '<div style="font-weight:700;margin-bottom:4px">Avance de caracterizacion</div>%s'
    '<div style="font-size:10px;color:#666;margin-top:4px">Intensidad = %% de avance '
    '(blanco 0%% a color 100%%). Gris = sin ciudadanos ubicados. '
    'Borde punteado = ubicacion mayoritariamente aproximada (snap al borde).</div></div>'
) % (BG, OKI_GRIS, leyenda_ejes)
m_av.get_root().html.add_child(folium.Element(legend_av))

# Capa del poligono de la zona de intervencion (Ciudad Paraiso), borde dorado.
if POLIGONO_ZONA is not None:
    zona_fg = folium.FeatureGroup(name="Poligono Ciudad Paraiso", show=True)
    folium.GeoJson(
        POLIGONO_ZONA.to_json(),
        style_function=lambda f: {"color": "#8A6D1A", "weight": 5,
                                  "dashArray": "10 6", "fill": False},
        tooltip="Zona de intervencion: Ciudad Paraiso",
    ).add_to(zona_fg)
    zona_fg.add_to(m_av)
folium.LayerControl(collapsed=False, position="topright").add_to(m_av)
html_av = os.path.join(OUT_DIR, "mapa_avance_caracterizacion.html")
m_av.save(html_av)
print("Mapa exportado:", html_av)
m_av


### 11f. Resumen analitico

In [ ]:
# 11f. Resumen analitico del avance de caracterizacion.
total_ciudadanos = int(len(U))
total_ubicados = int(U["lat"].notna().sum())
total_con_mz = int(U["IDMANZANA"].notna().sum())
pct_ubicados = round(100 * total_ubicados / total_ciudadanos, 1) if total_ciudadanos else 0.0
pct_con_mz = round(100 * total_con_mz / total_ciudadanos, 1) if total_ciudadanos else 0.0

# Avance promedio por eje sobre los ciudadanos con manzana asignada.
avance_prom_eje = {EJE_LABEL[e]: round(float(con_mz[e + "_sc"].mean() * 100), 1) if len(con_mz) else 0.0
                   for e in EJES}

resumen_av = {
    "ciudadanos_totales": total_ciudadanos,
    "ciudadanos_ubicados": total_ubicados,
    "pct_ubicados": pct_ubicados,
    "ciudadanos_con_manzana": total_con_mz,
    "pct_con_manzana": pct_con_mz,
    "manzanas_con_ciudadanos": int((mz_av["n_ciudadanos"] > 0).sum()),
    "avance_promedio_por_eje": avance_prom_eje,
}
print(json.dumps(resumen_av, ensure_ascii=False, indent=2))

MIN_CIUDADANOS = 3  # minimo para que el ranking no lo dominen manzanas con 1 registro
print("\nManzanas con MENOR avance global (con al menos %d ciudadanos):" % MIN_CIUDADANOS)
con_datos = mz_av[mz_av["n_ciudadanos"] >= MIN_CIUDADANOS].sort_values("pct_global").head(3)
if con_datos.empty:
    con_datos = mz_av[mz_av["n_ciudadanos"] > 0].sort_values("pct_global").head(3)
for _, r in con_datos.iterrows():
    print("  - Manzana %s (%s), Comuna %s Barrio %s: avance global %.0f%% | "
          "Ficha %.0f%% Hogares %.0f%% Salud %.0f%% Empresas %.0f%% | %d ciudadanos"
          % (r["MANZANA"], r["IDMANZANA"], r["COMUNA"], r["BARRIO"], r["pct_global"],
             r["pct_ficha"], r["pct_hogares"], r["pct_salud"], r["pct_empresas"],
             int(r["n_ciudadanos"])))

print("\nNota: el avance por manzana solo considera ciudadanos geolocalizados y")
print("asignados a una manzana catastral. Ciudadanos sin GPS ni direccion parseable")
print("no entran en la agregacion espacial. Datos reales de la encuesta (sin PII en el mapa).")
